In [1]:
import os
import sys
from tqdm import tqdm
import glob
import typing
import import_ipynb
import json

# Add current directory to path for imports
import os
current_dir = "/home2/ducvu/speech-processing-implement/codes"
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

import importlib
import config
from config import *
config_classifiers = config

import numpy as np
import pandas as pd
from collections import defaultdict

from pydub import AudioSegment
import numpy as np
from scipy import signal

In [2]:
BASE_DIR = config.BASE_DIR

audio_dir = f"{BASE_DIR}/raw_data/sub-1"
audio_files = glob.glob(f"{audio_dir}/*.wav")
slate_file = f"{BASE_DIR}/raw_data/slate-sound.wav"

In [3]:
print(audio_files)

['/home2/ducvu/speech-processing-implement//raw_data/sub-1/Sub-1_tech.wav', '/home2/ducvu/speech-processing-implement//raw_data/sub-1/sub-1_sub.wav']


In [4]:
def format_time(seconds):
    """Format seconds as MM:SS.mmm"""
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes:02d}:{secs:06.3f}"

In [5]:
def split_audio_by_slate(
                            audio_file:str, 
                            slate_file:str, 
                            threshold:int=0.6,
                            min_distance_sec:int=2,
                            save_json:bool=True,
                            output_json:str=f"{BASE_DIR}/processed_data/slate_positions",
                            file_name:str='',
                        ):
    
    # Loading audio file
    original_audio = AudioSegment.from_file(audio_file)
    slate = AudioSegment.from_file(slate_file)

    # Matching frame rate and channel
    if original_audio.frame_rate != slate.frame_rate:
        slate = slate.set_frame_rate(original_audio.frame_rate)

    if original_audio.channels == 2:
        original_audio = original_audio.set_channels(1)
    if slate.channels == 2:
        slate = slate.set_channels(1)
    # Convert to numpy array
    audio_array = np.array(original_audio.get_array_of_samples(), dtype=float)
    slate_array = np.array(slate.get_array_of_samples(), dtype=float)

    # Normalize audio
    audio_array = audio_array / np.max(np.abs(audio_array))
    slate_array = slate_array / np.max(np.abs(slate_array))

    # Cross-correlation
    correlation = signal.correlate(audio_array, slate_array, mode='valid')
    correlation = correlation / np.max(np.abs(correlation))

    # Find slate positions
    min_distance_samples = int(min_distance_sec * original_audio.frame_rate)

    slate_positions_samples = []
    correlation_values = []

    # Slate positions
    for i, corr_value in enumerate(correlation):
        if corr_value > threshold:
            if not slate_positions_samples or i - slate_positions_samples[-1] > min_distance_samples:
                slate_positions_samples.append(i)
                correlation_values.append(corr_value)

    # Convert to seconds
    slate_positions_seconds = [i / original_audio.frame_rate for i in slate_positions_samples]
    slate_positions_ms = [int(pos / original_audio.frame_rate * 1000) for pos in slate_positions_samples]

    # Create output folder
    os.makedirs(output_json, exist_ok=True)

    # Create result dictionary
    result = {
        "audio_file": audio_file,
        "slate_file": slate_file,
        "sample_rate": original_audio.frame_rate,
        "audio_duration_sec": len(audio_array) / original_audio.frame_rate,
        "audio_duration_ms": len(original_audio),
        "slate_duration_sec": len(slate_array) / original_audio.frame_rate,
        "slate_duration_ms": len(slate),
        "num_slates_found": len(slate_positions_samples),
        "threshold_used": threshold,
        "min_distance_sec": min_distance_sec,
        "slate_positions": [
            {
                "index": i,
                "sample": int(pos_sample),
                "time_sec": round(pos_sec, 3),
                "time_ms": pos_ms,
                "time_formatted": format_time(pos_sec),
                "correlation_score": round(corr, 4)
            }
            for i, (pos_sample, pos_sec, pos_ms, corr) in enumerate(
                zip(slate_positions_samples, slate_positions_seconds, 
                    slate_positions_ms, correlation_values)
            )
        ]
    }
    
    # Save to JSON
    if save_json:
        with open(f"{output_json}/{file_name}.json", 'w') as f:
            json.dump(result, f, indent=2)

    return result

In [6]:
def split_audio_by_positions(audio_file:str, 
                             positions_json:str="slate_positions.json",
                             output_folder:str="segments",
                             remove_slate:bool=True,
                             slate_buffer_ms:int=500,
                             min_segment_duration_sec:float=1.0,
                             file_name:str=''):

    
    # Load positions from JSON
    with open(positions_json, 'r') as f:
        data = json.load(f)
    
    
    # Load audio
    audio = AudioSegment.from_file(audio_file)
    
    # Extract slate positions in milliseconds
    slate_times_ms = [pos['time_ms'] for pos in data['slate_positions']]
    slate_duration_ms = data['slate_duration_ms']
    
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Calculate segment boundaries
    segments = []
    
    for i in range(len(slate_times_ms) + 1):
        if i == 0:
            # First segment (before first slate)
            start = 0
            end = slate_times_ms[0] if remove_slate else slate_times_ms[0] + slate_duration_ms
            label = "intro"
        elif i == len(slate_times_ms):
            # Last segment (after last slate)
            start = slate_times_ms[-1] + slate_duration_ms + slate_buffer_ms
            end = len(audio)
            label = "outro"
        else:
            # Middle segments (between slates)
            start = slate_times_ms[i-1] + slate_duration_ms + slate_buffer_ms
            end = slate_times_ms[i]
            if not remove_slate:
                end = end + slate_duration_ms
            label = f"segment"
        
        # Check minimum duration
        duration_sec = (end - start) / 1000.0
        if duration_sec >= min_segment_duration_sec:
            segments.append({
                'index': i,
                'start_ms': start,
                'end_ms': end,
                'duration_sec': duration_sec,
                'label': label
            })
    
    created_files = []
    
    for seg in segments:
        # Extract segment
        segment_audio = audio[seg['start_ms']:seg['end_ms']]
        
        
        output_path = os.path.join(output_folder, f"{file_name}_{seg['index']:03d}.wav")
        
        # Export
        segment_audio.export(output_path, format="wav")
        created_files.append(output_path)
        
    
    return created_files

In [7]:
for audio_file in audio_files:
    file_name = audio_file.split('/')[-1].split('.')[0]
    split_audio_by_slate(audio_file, slate_file, file_name=file_name)
    split_audio_by_positions(audio_file, f"{BASE_DIR}/processed_data/slate_positions/{file_name}.json", output_folder=f"{BASE_DIR}/processed_data/segments", file_name=file_name)